# Chapter 17: Export Systems and Fiscal Metering

**Production Optimization of Oil and Gas Fields Using NeqSim**

This notebook demonstrates export system modeling including:
- Gas export pipeline modeling with Beggs and Brill correlation
- Pressure and temperature profiles along the pipeline
- Gas quality calculations (heating value, Wobbe index concept)
- Sensitivity to pipeline length and diameter

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 1. Gas Export Pipeline Model

We model a subsea gas export pipeline from an offshore platform to
an onshore receiving terminal. The pipeline is 150 km long, 24-inch
diameter, with the Beggs and Brill multiphase correlation.

The export gas is a lean, processed natural gas after separation
and dehydration.

In [2]:
from neqsim import jneqsim

# Export gas composition (processed, dehydrated)
export_gas = jneqsim.thermo.system.SystemSrkEos(273.15 + 40.0, 180.0)
export_gas.addComponent("nitrogen", 1.5)
export_gas.addComponent("CO2", 2.0)
export_gas.addComponent("methane", 85.0)
export_gas.addComponent("ethane", 7.0)
export_gas.addComponent("propane", 3.0)
export_gas.addComponent("i-butane", 0.8)
export_gas.addComponent("n-butane", 0.7)
export_gas.setMixingRule("classic")

# Feed stream
feed = jneqsim.process.equipment.stream.Stream("Export Gas", export_gas)
feed.setFlowRate(5.0e6, "Sm3/day")  # 5 MSm3/day
feed.setTemperature(40.0, "C")
feed.setPressure(180.0, "bara")

# Export pipeline - 150 km, 24-inch, horizontal subsea
pipeline = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Export Pipeline", feed)
pipeline.setLength(150000.0)  # 150 km
pipeline.setElevation(0.0)    # horizontal (subsea)
pipeline.setDiameter(0.5588)  # 24 inch = 0.5588 m
pipeline.setPipeWallRoughness(4.5e-5)
pipeline.setNumberOfIncrements(30)
pipeline.setConstantSurfaceTemperature(4.0, "C")  # seabed temperature

process = jneqsim.process.processmodel.ProcessSystem()
process.add(feed)
process.add(pipeline)
process.run()

print("=== Gas Export Pipeline Results ===")
print(f"Pipeline length:       150 km")
print(f"Pipeline ID:           24 inch (558.8 mm)")
print(f"Inlet pressure:        {feed.getPressure('bara'):.1f} bara")
print(f"Outlet pressure:       {pipeline.getOutletStream().getPressure('bara'):.1f} bara")
print(f"Total pressure drop:   {pipeline.getPressureDrop():.1f} bar")
print(f"Inlet temperature:     {feed.getTemperature('C'):.1f} C")
print(f"Outlet temperature:    {pipeline.getOutletStream().getTemperature('C'):.1f} C")

=== Gas Export Pipeline Results ===
Pipeline length:       150 km
Pipeline ID:           24 inch (558.8 mm)
Inlet pressure:        180.0 bara
Outlet pressure:       177.2 bara
Total pressure drop:   2.8 bar
Inlet temperature:     40.0 C
Outlet temperature:    4.0 C


## 2. Pressure and Temperature Profiles Along Pipeline

The `PipeBeggsAndBrills` model computes profiles at each increment.
We extract and plot the pressure and temperature along the pipeline length.

In [3]:
# Extract profiles
pressure_profile = list(pipeline.getPressureProfile())
temperature_profile = list(pipeline.getTemperatureProfile())

# Convert temperature from K to C
temperature_profile_C = [t - 273.15 for t in temperature_profile]

# Distance along pipeline (km)
n_points = len(pressure_profile)
distance_km = np.linspace(0, 150, n_points)

# Plot pressure profile
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

ax1.plot(distance_km, pressure_profile, 'b-', linewidth=2)
ax1.set_ylabel("Pressure (bara)", fontsize=12)
ax1.set_title("Gas Export Pipeline: Pressure Profile", fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=pressure_profile[-1], color='r', linestyle='--', alpha=0.5,
            label=f"Arrival: {pressure_profile[-1]:.1f} bara")
ax1.legend(fontsize=11)

ax2.plot(distance_km, temperature_profile_C, 'r-', linewidth=2)
ax2.axhline(y=4.0, color='blue', linestyle='--', alpha=0.5, label="Seabed T = 4 C")
ax2.set_xlabel("Distance Along Pipeline (km)", fontsize=12)
ax2.set_ylabel("Temperature (C)", fontsize=12)
ax2.set_title("Gas Export Pipeline: Temperature Profile", fontsize=14)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig("../figures/ch17_pipeline_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ../figures/ch17_pipeline_profiles.png")

Figure saved to ../figures/ch17_pipeline_profiles.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_11892\2918442135.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Gas Quality Calculations

For fiscal metering and gas sales, key quality parameters include:
- **Gross Heating Value (GHV)** — energy content per unit volume
- **Wobbe Index** — interchangeability criterion for gas burners
- **Relative Density** (specific gravity)

We compute these from the thermodynamic properties of the export gas.

In [4]:
# Gas quality at standard conditions (15 C, 1.01325 bara)
gas_std = jneqsim.thermo.system.SystemSrkEos(273.15 + 15.0, 1.01325)
gas_std.addComponent("nitrogen", 1.5)
gas_std.addComponent("CO2", 2.0)
gas_std.addComponent("methane", 85.0)
gas_std.addComponent("ethane", 7.0)
gas_std.addComponent("propane", 3.0)
gas_std.addComponent("i-butane", 0.8)
gas_std.addComponent("n-butane", 0.7)
gas_std.setMixingRule("classic")

ops = jneqsim.thermodynamicoperations.ThermodynamicOperations(gas_std)
ops.TPflash()
gas_std.initProperties()

# Get molar mass and density
molar_mass = gas_std.getMolarMass() * 1000.0  # kg/kmol
gas_density = gas_std.getDensity("kg/m3")

# Air properties at standard conditions for relative density
air_molar_mass = 28.97  # kg/kmol
relative_density = molar_mass / air_molar_mass

# Component heating values (MJ/Sm3 at 15C, 1 atm) - standard reference values
heating_values = {
    "methane": 37.78,
    "ethane": 66.07,
    "propane": 93.94,
    "i-butane": 121.4,
    "n-butane": 121.8,
    "nitrogen": 0.0,
    "CO2": 0.0
}

# Mole fractions
composition = {
    "nitrogen": 1.5, "CO2": 2.0, "methane": 85.0,
    "ethane": 7.0, "propane": 3.0, "i-butane": 0.8, "n-butane": 0.7
}
total_mol = sum(composition.values())

# Gross heating value (volume-weighted)
ghv = sum((composition[c] / total_mol) * heating_values[c] for c in composition)

# Wobbe Index = GHV / sqrt(relative_density)
wobbe_index = ghv / np.sqrt(relative_density)

print("=== Gas Quality at Standard Conditions (15 C, 1.01325 bara) ===")
print(f"Molar mass:        {molar_mass:.2f} kg/kmol")
print(f"Gas density:       {gas_density:.4f} kg/m3")
print(f"Relative density:  {relative_density:.4f}")
print(f"Gross Heating Value: {ghv:.2f} MJ/Sm3")
print(f"Wobbe Index:       {wobbe_index:.2f} MJ/Sm3")
print()
print("Typical sales gas specification:")
print(f"  GHV:    36-42 MJ/Sm3  -> {'PASS' if 36 <= ghv <= 42 else 'FAIL'}")
print(f"  Wobbe:  46-52 MJ/Sm3  -> {'PASS' if 46 <= wobbe_index <= 52 else 'CHECK'}")

=== Gas Quality at Standard Conditions (15 C, 1.01325 bara) ===
Molar mass:        19.24 kg/kmol
Gas density:       0.8158 kg/m3
Relative density:  0.6640
Gross Heating Value: 41.38 MJ/Sm3
Wobbe Index:       50.78 MJ/Sm3

Typical sales gas specification:
  GHV:    36-42 MJ/Sm3  -> PASS
  Wobbe:  46-52 MJ/Sm3  -> PASS


## 4. Pipeline Diameter Sensitivity

Pipeline sizing is a critical design decision balancing CAPEX
(larger pipe = more material cost) against OPEX (smaller pipe = more
compression energy). We study pipe diameters from 18 to 36 inches.

In [5]:
# Pipe diameter sensitivity
diameters_inch = [18, 20, 22, 24, 26, 28, 30, 32, 34, 36]
diameters_m = [d * 0.0254 for d in diameters_inch]
pressure_drops = []
arrival_pressures = []
arrival_temperatures = []

for d_m in diameters_m:
    gas_i = jneqsim.thermo.system.SystemSrkEos(273.15 + 40.0, 180.0)
    gas_i.addComponent("nitrogen", 1.5)
    gas_i.addComponent("CO2", 2.0)
    gas_i.addComponent("methane", 85.0)
    gas_i.addComponent("ethane", 7.0)
    gas_i.addComponent("propane", 3.0)
    gas_i.addComponent("i-butane", 0.8)
    gas_i.addComponent("n-butane", 0.7)
    gas_i.setMixingRule("classic")

    feed_i = jneqsim.process.equipment.stream.Stream("Feed", gas_i)
    feed_i.setFlowRate(5.0e6, "Sm3/day")
    feed_i.setTemperature(40.0, "C")
    feed_i.setPressure(180.0, "bara")

    pipe_i = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills("Pipeline", feed_i)
    pipe_i.setLength(150000.0)
    pipe_i.setElevation(0.0)
    pipe_i.setDiameter(d_m)
    pipe_i.setPipeWallRoughness(4.5e-5)
    pipe_i.setNumberOfIncrements(20)
    pipe_i.setConstantSurfaceTemperature(4.0, "C")

    proc_i = jneqsim.process.processmodel.ProcessSystem()
    proc_i.add(feed_i)
    proc_i.add(pipe_i)
    proc_i.run()

    dp = pipe_i.getPressureDrop()
    p_arrival = pipe_i.getOutletStream().getPressure("bara")
    t_arrival = pipe_i.getOutletStream().getTemperature("C")

    pressure_drops.append(dp)
    arrival_pressures.append(p_arrival)
    arrival_temperatures.append(t_arrival)

print(f"{'Diameter (inch)':<18} {'dP (bar)':<12} {'P_arrival (bara)':<18} {'T_arrival (C)'}")
print("-" * 62)
for i, d in enumerate(diameters_inch):
    print(f"{d:<18} {pressure_drops[i]:<12.1f} {arrival_pressures[i]:<18.1f} {arrival_temperatures[i]:.1f}")

Diameter (inch)    dP (bar)     P_arrival (bara)   T_arrival (C)
--------------------------------------------------------------
18                 8.0          172.0              3.9
20                 4.6          175.4              4.0
22                 2.8          177.2              4.0
24                 1.8          178.2              4.0
26                 1.2          178.8              4.0
28                 0.8          179.2              4.0
30                 0.6          179.4              4.0
32                 0.4          179.6              4.0
34                 0.3          179.7              4.0
36                 0.2          179.8              4.0


In [6]:
# Plot: Pressure drop and arrival pressure vs diameter
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(diameters_inch, pressure_drops, 'o-', color="steelblue", linewidth=2, markersize=8)
ax1.set_xlabel("Pipeline Inner Diameter (inches)", fontsize=12)
ax1.set_ylabel("Total Pressure Drop (bar)", fontsize=12)
ax1.set_title("Pressure Drop vs Pipeline Diameter\n(150 km, 5 MSm3/day)", fontsize=13)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=50, color='r', linestyle='--', alpha=0.5, label="Typical max dP = 50 bar")
ax1.legend(fontsize=11)

ax2.plot(diameters_inch, arrival_pressures, 's-', color="seagreen", linewidth=2, markersize=8)
ax2.set_xlabel("Pipeline Inner Diameter (inches)", fontsize=12)
ax2.set_ylabel("Arrival Pressure (bara)", fontsize=12)
ax2.set_title("Arrival Pressure vs Pipeline Diameter\n(150 km, 5 MSm3/day)", fontsize=13)
ax2.grid(True, alpha=0.3)
ax2.axhline(y=70, color='r', linestyle='--', alpha=0.5, label="Min arrival P = 70 bara")
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig("../figures/ch17_diameter_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to ../figures/ch17_diameter_sensitivity.png")

Figure saved to ../figures/ch17_diameter_sensitivity.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_11892\2924853578.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Discussion

**Pressure Profile:** The pressure decreases nearly linearly along the pipeline
for single-phase gas flow. The slight non-linearity comes from gas expansion
reducing density (and velocity) along the length, which decreases frictional
losses per unit length downstream.

**Temperature Profile:** The gas cools from 40 C toward the seabed temperature
(4 C) as heat is lost through the pipe wall. The cooling is most rapid in the
first 30-40 km where the temperature difference is largest, following an
exponential decay pattern.

**Gas Quality:** The Gross Heating Value and Wobbe Index confirm the export gas
meets typical sales specifications. These parameters are critical for fiscal
metering and tariff calculations. CO2 and N2 are diluents that reduce heating
value and must be controlled.

**Pipeline Sizing:** Pressure drop is extremely sensitive to diameter — it
decreases roughly with the 5th power of diameter. Smaller pipelines (18-20")
may have unacceptable pressure losses for 5 MSm3/day throughput, while
larger pipelines (30-36") offer diminishing returns. The optimal size balances
CAPEX against the required inlet compression to maintain arrival pressure
above the minimum delivery specification.